# Setup env

In [7]:
import pandas as pd
from pathlib import Path

In [8]:
# Get the absolute path of where you are right now
src_path = Path.cwd()
project_src = src_path.parents[1]
print(f"Your project source root is: {project_src}")

# Set 100 rows to display
pd.set_option('display.max_rows', 100) # Set it higher than your 80 rows

Your project source root is: c:\Users\davib\Desktop\MSc_DataScience\DataWarehouse\lab_assignment\imbd


# Read Silver Layer

In [9]:
# Read dims
# bridge_profession_group = pd.read_parquet(f'{project_src}\\data\\silver\\bridge_profession_group.parquet')
# dim_profession = pd.read_parquet(f'{project_src}\\data\\silver\\dim_profession.parquet')
dim_person = pd.read_parquet(f'{project_src}\\data\\silver\\dim_person.parquet') ## <<<
# dim_roles = pd.read_parquet(f'{project_src}\\data\\silver\\dim_roles.parquet') ## <<<
# dim_genres = pd.read_parquet(f'{project_src}\\data\\silver\\dim_genres.parquet')
bridge_genres = pd.read_parquet(f'{project_src}\\data\\silver\\bridge_genres.parquet')
dim_title_basic = pd.read_parquet(f'{project_src}\\data\\silver\\dim_title_basic.parquet')
bridge_kwn_titles = pd.read_parquet(f'{project_src}\\data\\silver\\bridge_kwn_titles.parquet').dropna(subset=['tconst'])

# Read fact tables
participations_pers = pd.read_parquet(f'{project_src}\\data\\silver\\participations_pers.parquet') ## <<<

In [10]:
participations_pers.groupby('titleType').size()

C:\Users\davib\AppData\Local\Temp\ipykernel_14448\3114256188.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  participations_pers.groupby('titleType').size()


titleType
tvSeries    1133551
dtype: int64

In [4]:
# dim_person = pd.read_parquet(f'{project_src}\\data\\silver\\dim_person.parquet')
# dim_person

In [5]:
# dim_roles = pd.read_parquet(f'{project_src}\\data\\silver\\dim_roles.parquet')
# dim_roles

# Transformations to prepare final data for gold

Analysis to be done:
- Who are the most active people in the film industry? (more participations) OK
- What are the actors with more runtime done? (more runtime considering the participations) OK
- What are the participants which have worked in more distinct genres? OK

In [6]:
## Data Structure Exploration

In [11]:
# Explore key tables
print("participations_pers columns:", participations_pers.columns.tolist())
print("dim_person columns:", dim_person.columns.tolist())
print("dim_title_basic columns:", dim_title_basic.columns.tolist())
print("bridge_genres columns:", bridge_genres.columns.tolist())
print("bridge_kwn_titles columns:", bridge_kwn_titles.columns.tolist())
# print("dim_genres columns:", dim_genres.columns.tolist())
# print("dim_roles columns:", dim_roles.columns.tolist())

participations_pers columns: ['nconst', 'primaryName', 'tconst', 'titleType', 'primaryTitle', 'runtimeMinutes', 'genre_group_id', 'role_id', 'category', 'job', 'characters', 'profession_group_id', 'kwn_title_group_id', 'participation_count']
dim_person columns: ['nconst', 'primaryName', 'birthYear', 'deathYear', 'profession_group_id']
dim_title_basic columns: ['tconst', 'titleType', 'primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes', 'genre_group_id']
bridge_genres columns: ['tconst', 'genre_id', 'genre_group_id', 'weighting_factor_gen']
bridge_kwn_titles columns: ['tconst', 'nconst', 'kwn_title_group_id', 'weighting_factor_grp']


# Filter only movies and tvSeries

## Analysis 1: Most Active People (More Participations)

In [12]:
# Filter for relevant title types
# most_active_people = participations_pers[participations_pers['titleType'].isin(['movie', 'tvSeries'])]

# Count total participations per person and titleType
counts = (
    participations_pers.groupby(['nconst', 'titleType'])
    .size()
    .reset_index(name='total_participations')
)

# Sort by count and then take the top 10 for each titleType
top_10_per_type = (
    counts.sort_values(['titleType', 'total_participations'], ascending=[True, False])
    .groupby('titleType')
    .head(10)
    .merge(dim_person[['nconst', 'primaryName', 'birthYear', 'deathYear']], on='nconst', how='left')
)

top_10_per_type_known =top_10_per_type.merge(bridge_kwn_titles, on='nconst', how='left').merge(dim_title_basic[['tconst', 'primaryTitle', 'isAdult']], on='tconst', how='left')
top_10_per_type_known = top_10_per_type_known[['primaryName', 'total_participations', 'titleType', 'birthYear', 'deathYear', 'primaryTitle', 'isAdult']].dropna(subset=['primaryTitle'])
# Display the results
# print("Top 10 Most Active People per Category:")
# print(top_10_per_type[['titleType', 'primaryName', 'total_participations']])
# top_10_per_type_known.to_parquet(f'{project_src}\\data\\gold\\active_film_participants.parquet', index=False)

C:\Users\davib\AppData\Local\Temp\ipykernel_14448\4253797535.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  participations_pers.groupby(['nconst', 'titleType'])
C:\Users\davib\AppData\Local\Temp\ipykernel_14448\4253797535.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby('titleType')


In [13]:
top_10_per_type_known.head(80)

,primaryName,total_participations,titleType,birthYear,deathYear,primaryTitle,isAdult
1,Monica Rial,130,tvSeries,1975,<NA>,Gen: Lock,0.0
3,Don Messick,105,tvSeries,1926,1997,The Smurfs,0.0
4,Don Messick,105,tvSeries,1926,1997,The Jetsons,0.0
5,Tom Kenny,101,tvSeries,1962,<NA>,SpongeBob SquarePants,0.0
6,Mark Chinnery,99,tvSeries,1969,<NA>,Miracle Experiences! Unbelievable,0.0
7,Mark Chinnery,99,tvSeries,1969,<NA>,Catch the Wave Kodawari Navi,0.0
9,Dee Bradley Baker,88,tvSeries,1962,<NA>,Phineas and Ferb,0.0
10,Dee Bradley Baker,88,tvSeries,1962,<NA>,American Dad!,0.0
11,Rob Paulsen,88,tvSeries,1956,<NA>,Pinky and the Brain,0.0
12,Rob Paulsen,88,tvSeries,1956,<NA>,Tales of the Teenage Mutant Ninja Turtles,0.0


In [14]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Q12: Visualization - Top 10 Bar Chart per Title Type
# Prepare data for visualization
viz_data_q12 = (
    participations_pers.groupby(['nconst', 'titleType'])
    .size()
    .reset_index(name='total_participations')
    .sort_values(['titleType', 'total_participations'], ascending=[True, False])
    .groupby('titleType')
    .head(10)
    .merge(dim_person[['nconst', 'primaryName']], on='nconst', how='left')
)

# Create grouped bar chart
fig_q12 = px.bar(
    viz_data_q12,
    x='primaryName',
    y='total_participations',
    color='titleType',
    title='Q12: Top 10 Most Active Participants by Title Type',
    labels={'total_participations': 'Number of Participations', 'primaryName': 'Participant', 'titleType': 'Title Type'},
    barmode='group',
    height=500
)
fig_q12.update_xaxes(tickangle=-45)
fig_q12.show()

C:\Users\davib\AppData\Local\Temp\ipykernel_14448\2367777445.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  participations_pers.groupby(['nconst', 'titleType'])
C:\Users\davib\AppData\Local\Temp\ipykernel_14448\2367777445.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby('titleType')


## Analysis 2: Actors with More Runtime Done

In [15]:
# Actors with more runtime: Sum runtime from all participations
actors_runtime = (
    participations_pers
    .groupby('nconst')
    .agg({
        'runtimeMinutes': 'sum',
        'tconst': 'count'
    })
    .rename(columns={'runtimeMinutes': 'total_runtime_minutes', 'tconst': 'participation_count'})
    .reset_index()
    .sort_values('total_runtime_minutes', ascending=False, na_position='last')
    .merge(dim_person, on='nconst',  how='left')
    .dropna(subset=['total_runtime_minutes'])
    .head(20)
)
# 1. Merge all your data together
full_df = (
    actors_runtime[['primaryName', 'total_runtime_minutes', 'participation_count', 'birthYear', 'deathYear', 'nconst']]
    .merge(participations_pers[['nconst', 'tconst']], on='nconst', how='left')
    .merge(dim_title_basic[['tconst', 'primaryTitle', 'runtimeMinutes']], on='tconst', how='left')
)

# 2. Sort by nconst (the person) and runtimeMinutes (descending)
# 3. Group by nconst and take the top row for each
top_title_per_person = (
    full_df.sort_values(['nconst', 'runtimeMinutes'], ascending=[True, False])
    .groupby('nconst', as_index=False)
    .head(1)
    .reset_index(drop=True)
)
top_title_per_person = (
    top_title_per_person.sort_values('total_runtime_minutes', ascending=False)
    [['primaryName', 'primaryTitle', 'total_runtime_minutes', 'participation_count', 'birthYear', 'deathYear', 'primaryTitle', 'runtimeMinutes']]
    # .rename(columns={'primaryTitle': 'longestTitle', 'runtimeMinutes': 'longestTitleRuntimeMinutes'})
)
top_title_per_person = top_title_per_person.loc[:, ~top_title_per_person.columns.duplicated()]
# top_title_per_person.to_parquet(f'{project_src}\\data\\gold\\long_runtime_title.parquet', index=False) 

In [16]:
top_title_per_person

,primaryName,primaryTitle,total_runtime_minutes,participation_count,birthYear,deathYear,runtimeMinutes
9,Lisa Meeches,The Sharing Circle,8400.0,2,<NA>,<NA>,8400.0
0,Toshirô Mifune,Kôya no surônin,7035.0,6,1920,1997,2925.0
3,Peter Madden,In Town Tonight,5409.0,6,1904,1976,5220.0
6,Pauline Tooth,In Town Tonight,5220.0,2,1930,<NA>,5220.0
11,Joyce Bullen,In Town Tonight,5220.0,1,<NA>,<NA>,5220.0
19,John Ellison,In Town Tonight,5220.0,1,<NA>,<NA>,5220.0
4,Atsuo Nakamura,Kogarashi Monjirô,5097.0,11,1940,<NA>,1750.0
15,Metin Bilgin,Alim Dayi,4930.0,5,<NA>,<NA>,3000.0
16,Serap Oner,Alim Dayi,4900.0,4,<NA>,<NA>,3000.0
14,Salman Ahmad,Pepsi Top of the Pops,4853.0,2,<NA>,<NA>,2565.0


In [17]:
import plotly.graph_objects as go

# Q13: Visualization - Horizontal Bar Chart & Scatter Plot
# Prepare data for Q13
viz_data_q13 = (
    participations_pers
    .groupby('nconst')
    .agg({
        'runtimeMinutes': 'sum',
        'tconst': 'count'
    })
    .rename(columns={'runtimeMinutes': 'total_runtime_minutes', 'tconst': 'participation_count'})
    .reset_index()
    .sort_values('total_runtime_minutes', ascending=False, na_position='last')
    .merge(dim_person[['nconst', 'primaryName']], on='nconst', how='left')
    .dropna(subset=['total_runtime_minutes'])
    .head(20)
)

# Horizontal bar chart - Top 20 by Runtime
fig_q13_bar = px.bar(
    viz_data_q13,
    y='primaryName',
    x='total_runtime_minutes',
    orientation='h',
    title='Q13: Top 20 Participants by Total Accumulated Runtime',
    labels={'total_runtime_minutes': 'Total Runtime (minutes)', 'primaryName': 'Participant'},
    height=600
)
fig_q13_bar.show()

# Scatter plot - Runtime vs Participation Count
fig_q13_scatter = px.scatter(
    viz_data_q13,
    x='participation_count',
    y='total_runtime_minutes',
    color='total_runtime_minutes',
    size='participation_count',
    hover_name='primaryName',
    title='Q13: Runtime vs Participation Count',
    labels={'participation_count': 'Number of Participations', 'total_runtime_minutes': 'Total Runtime (minutes)'},
    height=500
)
fig_q13_scatter.show()

## Analysis 3: Participants with More Known Participations

In [18]:
# Participants with more known participations: using bridge_kwn_titles
# Count known titles per person (using bridge_kwn_titles if available)
known_participations = (
    bridge_kwn_titles.groupby('nconst')
    .size()
    .reset_index(name='known_title_count')
    .sort_values('known_title_count', ascending=False)
    .merge(dim_person, on='nconst', how='left')
    .head(20)
)

print("Top 20 Participants with Most Known Titles:")
print(known_participations)
known_participations

Top 20 Participants with Most Known Titles:
        nconst  known_title_count       primaryName  birthYear  deathYear  \
0    nm3251587                  4        Brian Fies       <NA>       <NA>   
1    nm1316066                  4       Erwin Tulfo       <NA>       <NA>   
2    nm0666168                  4     Dan Patterson       <NA>       <NA>   
3    nm3032649                  4     Nick Leighton       <NA>       <NA>   
4    nm0184526                  4     Bernard Cowan       <NA>       1990   
5    nm1551598                  4    Justin Roiland       1980       <NA>   
6    nm2069739                  4     Arnold Clavio       <NA>       <NA>   
7    nm0288621                  4      Derek Fowlds       1937       2020   
8    nm1642097                  4         Yu-ri Lee       1982       <NA>   
9    nm0679683                  4    Ashley Pharoah       1959       <NA>   
10   nm1580067                  4   Sandra Pinckney       <NA>       <NA>   
11   nm3310565                  

,nconst,known_title_count,primaryName,birthYear,deathYear,profession_group_id
0,nm3251587,4,Brian Fies,<NA>,<NA>,None
1,nm1316066,4,Erwin Tulfo,<NA>,<NA>,p_grp_2503574
2,nm0666168,4,Dan Patterson,<NA>,<NA>,p_grp_615224
3,nm3032649,4,Nick Leighton,<NA>,<NA>,p_grp_3862729
4,nm0184526,4,Bernard Cowan,<NA>,1990,p_grp_171583
5,nm1551598,4,Justin Roiland,1980,<NA>,p_grp_2698789
6,nm2069739,4,Arnold Clavio,<NA>,<NA>,p_grp_3117813
7,nm0288621,4,Derek Fowlds,1937,2020,p_grp_267730
8,nm1642097,4,Yu-ri Lee,1982,<NA>,p_grp_2766638
9,nm0679683,4,Ashley Pharoah,1959,<NA>,p_grp_627625


In [19]:
# Q14: Visualization - Scatter Plot & Correlation
# Prepare correlation data
actual_participations = (
    participations_pers.groupby('nconst')
    .size()
    .reset_index(name='actual_participation_count')
)

correlation_data = (
    known_participations[['nconst', 'known_title_count', 'primaryName']]
    .merge(actual_participations, on='nconst', how='inner')
)

# Scatter plot - Known vs Actual Participations
fig_q14_scatter = px.scatter(
    correlation_data,
    x='actual_participation_count',
    y='known_title_count',
    hover_name='primaryName',
    trendline='ols',
    title='Q14: Known Titles vs Actual Participations (Correlation)',
    labels={'actual_participation_count': 'Actual Participations', 'known_title_count': 'Known Titles'},
    height=500
)
fig_q14_scatter.show()

# Calculate correlation coefficient
correlation_coeff = correlation_data['actual_participation_count'].corr(correlation_data['known_title_count'])
print(f"Correlation between Known Titles and Actual Participations: {correlation_coeff:.3f}")

ModuleNotFoundError: No module named 'statsmodels'

## Analysis 4: Participants with More Distinct Genres

In [20]:
# Participants with more distinct genres worked with
# Join participations with genres through title
participants_genres = (
    participations_pers.merge(bridge_genres, on='tconst', how='inner')
    .groupby('nconst')['genre_id']
    .nunique()
    .reset_index(name='distinct_genres_count')
    .sort_values('distinct_genres_count', ascending=False)
    .merge(dim_person[[ 'primaryName' , 'birthYear', 'deathYear', 'nconst']], on='nconst', how='left')
    .head(20)
)

# print("Top 20 Participants with Most Distinct Genres:")
# print(participants_genres)
participants_genres

,nconst,distinct_genres_count,primaryName,birthYear,deathYear
0,nm0001495,22,Patrick Macnee,1922,2015
1,nm0001637,21,Vincent Price,1911,1993
2,nm0294067,18,Dawn French,1957,<NA>
3,nm2045573,18,Michael Q. Schmidt,1953,<NA>
4,nm0000799,18,Edward Asner,1929,<NA>
5,nm0000608,18,Burt Reynolds,1936,2018
6,nm0694293,17,Robert Powell,1944,<NA>
7,nm0000638,17,William Shatner,1931,<NA>
8,nm0001290,17,Richard E. Grant,1957,<NA>
9,nm0097773,17,Lydia Bosch,1963,<NA>


In [21]:
# Q15: Visualization - Horizontal Bar Chart & Bubble Chart
# Prepare data for Q15
viz_data_q15 = (
    participations_pers.merge(bridge_genres, on='tconst', how='inner')
    .groupby('nconst')
    .agg({
        'genre_id': 'nunique',
        'tconst': 'count'
    })
    .reset_index()
    .rename(columns={'genre_id': 'distinct_genres_count', 'tconst': 'participation_count'})
    .sort_values('distinct_genres_count', ascending=False)
    .merge(dim_person[['nconst', 'primaryName']], on='nconst', how='left')
    .head(20)
)

# Horizontal bar chart
fig_q15_bar = px.bar(
    viz_data_q15,
    y='primaryName',
    x='distinct_genres_count',
    orientation='h',
    title='Q15: Top 20 Participants by Distinct Genres',
    labels={'distinct_genres_count': 'Number of Distinct Genres', 'primaryName': 'Participant'},
    height=600
)
fig_q15_bar.show()

# Bubble chart - Genres vs Participations
fig_q15_bubble = px.scatter(
    viz_data_q15,
    x='participation_count',
    y='distinct_genres_count',
    size='distinct_genres_count',
    color='distinct_genres_count',
    hover_name='primaryName',
    title='Q15: Genre Diversity vs Participation Count',
    labels={'participation_count': 'Total Participations', 'distinct_genres_count': 'Distinct Genres'},
    height=500
)
fig_q15_bubble.show()

## Analysis 5: Collaboration & Network Patterns

In [ ]:
### 5a: Strongest Collaborations (Participants in Most Titles Together)
# Find pairs of participants who appear in the most titles together
from itertools import combinations

# Get pairs of participants per title
title_participants = participations_pers[['tconst', 'nconst']].drop_duplicates()

# Create collaboration pairs
collaborations = []
for tconst, group in title_participants.groupby('tconst'):
    participants = group['nconst'].tolist()
    if len(participants) > 1:
        # Create all pairs
        pairs = list(combinations(sorted(participants), 2))
        for pair in pairs:
            collaborations.append({'participant_1': pair[0], 'participant_2': pair[1], 'tconst': tconst})

collab_df = pd.DataFrame(collaborations)

# Count collaboration frequency
strongest_collabs = (
    collab_df.groupby(['participant_1', 'participant_2'])
    .agg({'tconst': 'count'})
    .reset_index()
    .rename(columns={'tconst': 'shared_titles_count'})
    .sort_values('shared_titles_count', ascending=False)
    .head(30)
)

# Add participant names
strongest_collabs = strongest_collabs.merge(
    dim_person[['nconst', 'primaryName']].rename(columns={'nconst': 'participant_1', 'primaryName': 'person_1_name'}),
    on='participant_1', how='left'
).merge(
    dim_person[['nconst', 'primaryName']].rename(columns={'nconst': 'participant_2', 'primaryName': 'person_2_name'}),
    on='participant_2', how='left'
)

strongest_collabs_display = strongest_collabs[['person_1_name', 'person_2_name', 'shared_titles_count']]
strongest_collabs_display.to_parquet(f'{project_src}\\data\\gold\\strongest_collaborations.parquet', index=False)
print("Top 30 Strongest Collaborations:")
strongest_collabs_display.head(30)

In [ ]:
# Q16: Visualization - Top Collaborations Table
# Display the top 30 collaborations as interactive table
fig_q16_table = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>Person 1</b>', '<b>Person 2</b>', '<b>Shared Titles</b>'],
        fill_color='paleturquoise',
        align='left',
        font=dict(color='black', size=12)
    ),
    cells=dict(
        values=[
            strongest_collabs_display['person_1_name'].head(30),
            strongest_collabs_display['person_2_name'].head(30),
            strongest_collabs_display['shared_titles_count'].head(30)
        ],
        fill_color='lavender',
        align='left',
        font=dict(color='black', size=11)
    )
)])

fig_q16_table.update_layout(
    title='Q16: Top 30 Strongest Collaborations (Pairs with Most Shared Titles)',
    height=600
)
fig_q16_table.show()

# Bar chart of top collaborations
top_collabs_viz = strongest_collabs_display.head(15).copy()
top_collabs_viz['collaboration'] = top_collabs_viz['person_1_name'] + ' & ' + top_collabs_viz['person_2_name']

fig_q16_bar = px.bar(
    top_collabs_viz,
    y='collaboration',
    x='shared_titles_count',
    orientation='h',
    title='Q16: Top 15 Strongest Collaborations',
    labels={'shared_titles_count': 'Number of Shared Titles', 'collaboration': 'Collaboration Pair'},
    height=600
)
fig_q16_bar.show()

## Analysis 6: Decade-Based Analysis

In [ ]:
### 6a: Decades with Most Active Participants
# Extract decade from startYear in participations_pers
# First, check if startYear exists or use another time indicator

# Join with dim_title_basic to get year information
participations_with_year = (
    participations_pers.merge(dim_title_basic[['tconst', 'startYear']], on='tconst', how='left')
    .dropna(subset=['startYear'])
)

# Convert year to decade
participations_with_year['decade'] = (participations_with_year['startYear'] // 10 * 10).astype(int)

# Count active participants (unique nconst) per decade
active_participants_per_decade = (
    participations_with_year.groupby('decade')['nconst']
    .nunique()
    .reset_index(name='unique_participants')
    .sort_values('decade')
)

# Also count total participations per decade
total_participations_per_decade = (
    participations_with_year.groupby('decade')
    .size()
    .reset_index(name='total_participations')
)

decades_summary = active_participants_per_decade.merge(total_participations_per_decade, on='decade')
decades_summary['avg_participations_per_person'] = (
    decades_summary['total_participations'] / decades_summary['unique_participants']
).round(2)

# decades_summary.to_parquet(f'{project_src}\\data\\gold\\decades_participant_activity.parquet', index=False)
print("Participant Activity by Decade:")
decades_summary

Participant Activity by Decade:


,decade,unique_participants,total_participations,avg_participations_per_person
0,1970,10,10,1.00
1,1980,12,13,1.08
2,1990,225,230,1.02
3,2000,3368,4894,1.45
4,2010,3800,6361,1.67
5,2020,122,152,1.25


In [ ]:
# Q17: Visualization - Decade Trends
from plotly.subplots import make_subplots

# Line chart - Activity trends across decades
fig_q17_line = px.line(
    decades_summary,
    x='decade',
    y='unique_participants',
    title='Q17a: Unique Participants by Decade (Industry Growth Trend)',
    labels={'unique_participants': 'Number of Unique Participants', 'decade': 'Decade'},
    markers=True,
    height=500
)
fig_q17_line.show()

# Dual-axis chart - Unique Participants & Total Participations
fig_q17_dual = make_subplots(specs=[[{"secondary_y": True}]])

fig_q17_dual.add_trace(
    px.line(decades_summary, x='decade', y='unique_participants').data[0],
    secondary_y=False,
)

fig_q17_dual.add_trace(
    px.line(decades_summary, x='decade', y='total_participations', 
            line_dash='dash').data[0],
    secondary_y=True,
)

fig_q17_dual.update_layout(
    title='Q17b: Dual-Axis: Unique Participants (solid) vs Total Participations (dashed)',
    xaxes_title='Decade',
    height=500
)

fig_q17_dual.update_yaxes(title_text='<b>Unique Participants</b>', secondary_y=False)
fig_q17_dual.update_yaxes(title_text='<b>Total Participations</b>', secondary_y=True)

fig_q17_dual.show()

# Area chart showing participation growth
fig_q17_area = px.area(
    decades_summary,
    x='decade',
    y='total_participations',
    title='Q17c: Total Participations Growth Over Decades',
    labels={'total_participations': 'Total Participations', 'decade': 'Decade'},
    height=500
)
fig_q17_area.show()

ValueError: Value of 'line_dash' is not the name of a column in 'data_frame'. Expected one of ['decade', 'unique_participants', 'total_participations', 'avg_participations_per_person'] but received: dash

In [ ]:
### 6b: New Entrants per Decade
# Find the first appearance year for each participant
first_appearance = (
    participations_with_year.groupby('nconst')['startYear']
    .min()
    .reset_index(name='first_year')
)

first_appearance['entry_decade'] = (first_appearance['first_year'] // 10 * 10).astype(int)

# Count new entrants per decade
new_entrants_per_decade = (
    first_appearance.groupby('entry_decade')
    .size()
    .reset_index(name='new_entrants')
    .sort_values('entry_decade')
)

# new_entrants_per_decade.to_parquet(f'{project_src}\\data\\gold\\new_entrants_per_decade.parquet', index=False)
print("New Participants Entering per Decade:")
new_entrants_per_decade

New Participants Entering per Decade:


,entry_decade,new_entrants
0,1900,10
1,1910,10
2,1920,20
3,1930,372
4,1940,2244
5,1950,13867
6,1960,20329
7,1970,23514
8,1980,33088
9,1990,54916


In [ ]:
### 6c: Career Peak Detection - Most Active Decade per Participant
# Count participations per person per decade
participations_per_decade = (
    participations_with_year.groupby(['nconst', 'decade'])
    .size()
    .reset_index(name='participations_in_decade')
)

# Find peak decade for each participant
career_peak = (
    participations_per_decade.sort_values('participations_in_decade', ascending=False)
    .groupby('nconst')
    .head(1)
    .reset_index(drop=True)
)

# Add participant details
career_peak = career_peak.merge(
    dim_person[['nconst', 'primaryName', 'birthYear', 'deathYear']], 
    on='nconst', 
    how='left'
)

# Calculate career span
career_peak_sorted = (
    participations_per_decade.merge(dim_person[['nconst', 'primaryName']], on='nconst', how='left')
    .sort_values(['nconst', 'decade'])
)

career_span = (
    career_peak_sorted.groupby('nconst')
    .agg({'decade': ['min', 'max']})
    .reset_index()
)

career_span.columns = ['nconst', 'first_decade', 'last_decade']
career_span['career_span_decades'] = career_span['last_decade'] - career_span['first_decade']

career_peak_final = career_peak.merge(
    career_span[['nconst', 'first_decade', 'last_decade', 'career_span_decades']], 
    on='nconst', 
    how='left'
).sort_values('participations_in_decade', ascending=False).head(25)

career_peak_display = career_peak_final[['primaryName', 'decade', 'participations_in_decade', 'first_decade', 'last_decade', 'career_span_decades']]
# career_peak_display.to_parquet(f'{project_src}\\data\\gold\\career_peak_by_decade.parquet', index=False)
print("Top 25 Participants by Career Peak (Most Active Decade):")
career_peak_display.head(25)

Top 25 Participants by Career Peak (Most Active Decade):


,primaryName,decade,participations_in_decade,first_decade,last_decade,career_span_decades
0,Taissia Shanti,2010,80,2000,2010,10
1,Andrey Da!,2010,73,2000,2010,10
2,Monica Rial,2010,72,1980,2020,40
3,Luci Christian,2000,68,1990,2020,30
4,Kana Hanazawa,2010,67,2000,2020,20
5,Brittney Karbowski,2010,57,2000,2020,20
6,Eric Whiteley,2010,56,2010,2020,10
7,Aaron Elliott,2010,56,2010,2020,10
8,Mark Chinnery,2010,53,1980,2020,40
9,Shane Farley,2010,53,2010,2020,10


In [ ]:
# Q18: Visualization - Career Peak & Span Analysis
# Prepare full dataset for visualizations
career_stats = (
    participations_per_decade.merge(dim_person[['nconst', 'primaryName']], on='nconst', how='left')
    .sort_values(['nconst', 'decade'])
)

career_span_full = (
    career_stats.groupby('nconst')
    .agg({'decade': ['min', 'max']})
    .reset_index()
)

career_span_full.columns = ['nconst', 'first_decade', 'last_decade']
career_span_full['career_span_decades'] = career_span_full['last_decade'] - career_span_full['first_decade']

# Find peak decade for each participant
career_peak_full = (
    participations_per_decade.sort_values('participations_in_decade', ascending=False)
    .groupby('nconst')
    .head(1)
    .reset_index(drop=True)
    .merge(career_span_full, on='nconst', how='left')
    .merge(dim_person[['nconst', 'primaryName']], on='nconst', how='left')
)

# Scatter plot - Peak Decade vs Career Span
fig_q18_scatter = px.scatter(
    career_peak_full.head(100),
    x='career_span_decades',
    y='decade',
    size='participations_in_decade',
    color='participations_in_decade',
    hover_name='primaryName',
    title='Q18a: Career Peak Decade vs Career Span (bubble size = peak participations)',
    labels={'career_span_decades': 'Career Span (decades)', 'decade': 'Peak Decade', 'participations_in_decade': 'Participations in Peak Decade'},
    height=500
)
fig_q18_scatter.show()

# Histogram - Distribution of Peak Decades
fig_q18_hist = px.histogram(
    career_peak_full,
    x='decade',
    nbins=15,
    title='Q18b: Distribution of Career Peak Decades',
    labels={'decade': 'Decade', 'count': 'Number of Participants'},
    height=500
)
fig_q18_hist.show()

# Bar chart - Career Span Distribution
career_span_dist = (
    career_span_full['career_span_decades']
    .value_counts()
    .sort_index()
    .reset_index()
)
career_span_dist.columns = ['career_span_decades', 'count']

fig_q18_span = px.bar(
    career_span_dist.head(20),
    x='career_span_decades',
    y='count',
    title='Q18c: Distribution of Career Spans (in decades)',
    labels={'career_span_decades': 'Career Span (decades)', 'count': 'Number of Participants'},
    height=500
)
fig_q18_span.show()

print("\nCareer Statistics Summary:")
print(f"Average Career Span: {career_span_full['career_span_decades'].mean():.1f} decades")
print(f"Median Career Span: {career_span_full['career_span_decades'].median():.1f} decades")
print(f"Max Career Span: {career_span_full['career_span_decades'].max()} decades")
print(f"Most Common Peak Decade: {career_peak_full['decade'].mode().values[0]}")
